[![img/pythonista.png](img/pythonista.png)](https://www.pythonista.io)

# Tendencia central y dispersión.

**py401 · EDA Práctico** · Pythonista® · Instructor: José Luis Chiquete · Módulo II: *Describir y comparar*

> 📝 **Versión del alumno.** Las celdas marcadas con `# TU CÓDIGO AQUÍ` y las de *Tu respuesta* son para ti.

En este *notebook* se describirán distribuciones con medidas de tendencia central y de dispersión, clásicas y robustas (media, mediana, desviación estándar, IQR y MAD), usando `penguins_limpio`.

## 🧭 Cómo usar este *notebook*.

* Ejecuta las celdas **en orden** (`Shift + Enter`).
* 🟢 **Ejercicio básico:** lo que todos deben dominar.
* 🔴 **Reto (opcional):** para quien termina antes o quiere profundizar. Si no da tiempo en clase, se puede resolver como práctica.
* Las celdas `# TU CÓDIGO AQUÍ` y *✍️ Tu respuesta* son para ti.

## 📚 Referencia de la API: curso py311.

Esta sesión usa métodos ya vistos en py311 sin repasarlos uno por uno: `quantile`, `value_counts` y `nlargest` en [py311 · 19 Herramientas de análisis de datos](https://github.com/PythonistaMX/py311/blob/main/19_herramientas_de_analisis_de_datos.ipynb); `transform` y `apply` en [py311 · 17 Los métodos `apply()` y `transform()`](https://github.com/PythonistaMX/py311/blob/main/17_metodos_apply_y_transform.ipynb); `groupby` en [py311 · 24 Métodos `groupby()`](https://github.com/PythonistaMX/py311/blob/main/24_metodos_groupby.ipynb); y las gráficas de distribución de *Seaborn* en [py311 · 31 Introducción a *Seaborn*](https://github.com/PythonistaMX/py311/blob/main/31_introduccion_a_seaborn.ipynb). El tiempo de la sesión se dedica a **decidir** qué medida describe mejor cada distribución y qué hacer con los atípicos.

> ⚠️ **Nota:** py311 se escribió para *pandas* 2 y este curso usa *pandas* 3, así que pueden verse diferencias (por ejemplo, texto como `object` en lugar de `str`, o el uso de `inplace=True`). Cuando difieran, se sigue lo que muestra este *notebook* (ver la sección 1.1 de `03_pandas_3_esencial.ipynb`).

### Preparación del entorno.

* La siguiente celda importará las bibliotecas, incluido el módulo `stats` de *SciPy*, y definirá `DATA` y la semilla `rng`.

In [ ]:
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

DATA = next(os.path.join(p, "data") for p in (os.curdir, os.pardir)
            if os.path.isfile(os.path.join(p, "data", "penguins.csv")))
pd.set_option("display.max_columns", 20)
sns.set_theme(style="whitegrid")
rng = np.random.default_rng(401)
from scipy import stats

warnings.filterwarnings("ignore", message="vert: bool was deprecated")

**Nota:** el filtro de advertencias oculta solo un aviso interno de *Seaborn* 0.13 con *Matplotlib* 3.11 al dibujar diagramas de caja (`vert: bool was deprecated`). No afecta los resultados.

* La siguiente celda leerá `penguins_limpio.csv` o lo regenerará desde `penguins.csv` si no existe.

In [ ]:
ruta_limpio = os.path.join(DATA, "penguins_limpio.csv")
medidas = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
if not os.path.isfile(ruta_limpio):
    (
        pd.read_csv(os.path.join(DATA, "penguins.csv"))
        .assign(sex=lambda d: d["sex"].str.title())
        .dropna(subset=medidas, how="all")
        .to_csv(ruta_limpio, index=False)
    )
pinguinos = pd.read_csv(ruta_limpio).astype({"species": "category", "island": "category", "sex": "category"})
pinguinos.shape

---
## 1. Repaso: `penguins_limpio` (10 min).

En la sesión 2 se generó `data/penguins_limpio.csv`: `sex` homologado a `Male`/`Female`, sin las 2 filas que no tenían medidas y con `sex` faltante sin imputar. Si el archivo no existe (por ejemplo, porque no se ejecutó `09_auditoria_de_calidad.ipynb`), se regenera con las mismas decisiones.

* La siguiente celda mostrará los tipos y los faltantes que quedan.

In [ ]:
pd.DataFrame({"dtype": pinguinos.dtypes.astype(str), "faltantes": pinguinos.isna().sum()})

---
## 2. Tendencia central y dispersión (25 min).

| Medida | Qué resume | ¿Robusta? | En código |
|---|---|---|---|
| Media | Centro de gravedad | No: un solo valor extremo la arrastra | `s.mean()` |
| Mediana | Valor que deja la mitad de los datos a cada lado | Sí | `s.median()` |
| Desviación estándar | Distancia típica a la media | No | `s.std()` (con `ddof=1`) |
| IQR (rango intercuartílico) | Ancho del 50% central: del valor que deja abajo al 25% de los datos (primer cuartil) al que deja abajo al 75% (tercer cuartil) | Sí | `stats.iqr(s)` |
| MAD (desviación absoluta mediana) | Mediana de las distancias de cada valor a la mediana | Sí | `stats.median_abs_deviation(s, scale="normal")` |

Una medida es **robusta** si no cambia mucho cuando una pequeña parte de los datos es extrema o errónea. Con `scale="normal"`, la MAD se multiplica por 1.4826 para que sea comparable con la desviación estándar cuando los datos tienen forma de campana (distribución normal).

**Concepto: cómo se obtiene cada medida, en palabras.**
* **Media:** se suman todos los valores y se divide entre cuántos son.
* **Mediana:** se ordenan los valores y se toma el del medio (si hay un número par de valores, el promedio de los dos del medio).
* **Desviación estándar:** resume qué tan lejos de la media suelen estar los valores. Se mide la distancia de cada valor a la media, se elevan al cuadrado (para que las distancias hacia arriba y hacia abajo no se cancelen), se promedian y se saca raíz cuadrada para regresar a las unidades originales. *pandas* divide entre el número de datos menos uno (`ddof=1`), un ajuste que importa sobre todo con muestras pequeñas.
* **IQR:** el tercer cuartil menos el primero; es decir, el ancho del 50% central de los datos.
* **MAD:** la mediana de las distancias de cada valor a la mediana.

**Ejemplo:** con las masas 3,000, 3,500, 3,600, 3,700 y 9,000 g, la media es 4,560 g y la mediana, 3,600 g. El valor de 9,000 g arrastró la media casi 1,000 g hacia arriba; la mediana ni se enteró.

La media y la desviación estándar usan **todos los valores con su magnitud**, por eso un valor extremo las arrastra; la mediana, el IQR y la MAD solo dependen del **orden** de los valores centrales. En el flujo del EDA, describir el centro y la dispersión es el primer resumen de cada variable clave, y la forma (sección 1 de `11_forma_de_la_distribucion.ipynb`) decide cuál de las dos parejas reportar.

* La siguiente celda extraerá la masa de los Adelie.

In [ ]:
masa_adelie = pinguinos.loc[pinguinos["species"] == "Adelie", "body_mass_g"]
masa_adelie.describe()

* La siguiente celda calculará las cinco medidas para la masa de los Adelie.

In [ ]:
pd.Series({
    "media": masa_adelie.mean(),
    "mediana": masa_adelie.median(),
    "desv_est": masa_adelie.std(),
    "iqr": stats.iqr(masa_adelie),
    "mad": stats.median_abs_deviation(masa_adelie, scale="normal"),
}).round(1)

**Interpretación:** media (3,700.7 g) y mediana (3,700 g) casi coinciden, y la desviación estándar (458.6 g) y la MAD (444.8 g) también: es la firma de una distribución **casi simétrica y sin atípicos**. En ese caso ambas parejas dicen lo mismo y se puede reportar media y desviación estándar. El IQR (650 g) es mayor que la desviación estándar porque mide el ancho del 50% central, no una "distancia típica": en datos con forma de campana, el IQR es aproximadamente 1.35 veces la desviación estándar (aquí, 650 / 458.6 ≈ 1.4).

**Ejemplo:** se simulará un error de captura: alguien escribe 37,500 g en lugar de 3,750 g. Se compararán las medidas con y sin ese único valor.

* La siguiente celda agregará el valor erróneo y comparará las medidas.

In [ ]:
con_error = pd.concat([masa_adelie, pd.Series([37500])], ignore_index=True)

def medidas_basicas(serie):
    return pd.Series({
        "media": serie.mean(),
        "mediana": serie.median(),
        "desv_est": serie.std(),
        "iqr": stats.iqr(serie),
        "mad": stats.median_abs_deviation(serie, scale="normal"),
    })

pd.DataFrame({"original": medidas_basicas(masa_adelie), "con un error": medidas_basicas(con_error)}).round(1)

Un solo valor erróneo entre 152 sube la media en más de 200 g (de 3,701 a 3,923 g) y multiplica por seis la desviación estándar (de 459 a 2,779 g), mientras que la mediana, el IQR y la MAD no se mueven. Por eso, ante datos sospechosos, conviene reportar **mediana e IQR** junto con media y desviación estándar.

### 🟢 Ejercicio 1 · Una función de resumen.

**🎯 Objetivo:** escribir una función que reúna medidas sensibles y robustas. **Qué se espera aprender:** a reportar siempre centro y dispersión en pareja, y a reutilizar el código en lugar de copiarlo.

Escribe `resumen(serie)` que devuelva una `Series` con `n` (no nulos), `media`, `mediana`, `desv_est`, `iqr` y `mad` (con `scale="normal"`), ignorando los faltantes. Aplícala a `body_mass_g` y a `flipper_length_mm` de todos los pingüinos y muestra ambos resultados en un *dataframe*.

* La siguiente celda es para tu solución.

In [ ]:
# TU CÓDIGO AQUÍ
def resumen(serie):
    return None

### 🟢 Ejercicio 2 · Descripción por especie.

**🎯 Objetivo:** aplicar el resumen dentro de cada grupo. **Qué se espera aprender:** que una descripción global puede ocultar grupos con centros y dispersiones muy distintos.

Describe `body_mass_g` y `flipper_length_mm` **por especie** con las mismas medidas (pista: `groupby("species")[columna].apply(resumen).unstack()`, o `agg` con una lista de funciones). Compara con el resumen global del ejercicio 1: ¿qué medida global describe mal a **todas** las especies?

* La siguiente celda es para tu solución.

In [ ]:
# TU CÓDIGO AQUÍ

**✍️ Tu respuesta:**

_(escribe aquí)_

### 🔴 Reto 1 · Punto de quiebre.

**🎯 Objetivo:** medir cuánta contaminación tolera cada medida. **Qué se espera aprender:** el concepto de punto de quiebre, que explica por qué la mediana es robusta y la media no.

Toma la masa de los Gentoo y reemplaza una proporción creciente de valores (0%, 5%, 10%, …, 45%) por un valor absurdo (60,000 g). Para cada proporción calcula media, mediana, desviación estándar y MAD, y grafica cómo cambian. ¿A partir de qué proporción deja de servir la mediana? (El **punto de quiebre** de la media es 0%; el de la mediana, 50%.)

* La siguiente celda es para tu solución.

In [ ]:
# TU CÓDIGO AQUÍ (opcional)

**✍️ Tu respuesta:**

_(escribe aquí)_

📖 **Para profundizar (opcional).** *Las fórmulas de la sección 2.* Con los datos ordenados de menor a mayor, $x_{(1)} \leq \dots \leq x_{(n)}$, y con $\tilde{x}$ como la mediana:

$$\bar{x} = \frac{1}{n}\sum_{i=1}^{n} x_i \qquad s = \sqrt{\frac{1}{n-1}\sum_{i=1}^{n}(x_i - \bar{x})^2} \qquad IQR = Q_3 - Q_1 \qquad MAD = 1.4826 \cdot \operatorname{mediana}\left(|x_i - \tilde{x}|\right)$$

Dividir entre $n - 1$ en lugar de $n$ hace que la **varianza** ($s^2$) no subestime en promedio a la de la población.

**💡 Ideas clave de la sección 2:**
* Media y desviación estándar usan la magnitud de todos los valores; mediana, IQR y MAD solo el orden: son **robustas**.
* Un solo error puede mover mucho la media y la desviación estándar.
* Se reporta la pareja adecuada a la forma, o ambas con justificación.

---
## ✅ Cierre.

**Práctica (sin calificación, en Classroom):** terminar los ejercicios 🟢 pendientes de este *notebook* y, si es posible, un reto 🔴.

**Siguiente *notebook*:** `11_forma_de_la_distribucion.ipynb`.

<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Licencia Creative Commons" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/80x15.png" /></a><br />Esta obra está bajo una <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2017-2026.</p>